In [92]:
!pip install qiskit google-generativeai python-dotenv matplotlib

In [93]:
import sys
!{sys.executable} -m pip install pennylane torch torchvision

In [94]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

def train_mnist(lr=0.01, epochs=2):
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimpleClassifier().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, (data, target) in enumerate(loader):
            if batch_idx > 20: break 
            
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        avg_loss = running_loss / 21
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")
        
    return model

mnist_model = train_mnist(lr=0.01, epochs=2)

Epoch 1/2 | Loss: 1.1174
Epoch 2/2 | Loss: 0.4140


In [95]:
!pip install kaggle
!kaggle datasets download -d vishakkbhat/ml4sci

Dataset URL: https://www.kaggle.com/datasets/vishakkbhat/ml4sci
License(s): unknown
ml4sci.zip: Skipping, found more recently modified local copy (use --force to force download)


In [96]:
import zipfile
import os

zip_path = 'ml4sci.zip'
extract_folder = 'quark_gluon_data'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
    print("Extraction complete.")
else:
    print("Error: ml4sci.zip not found in the current directory.")
    
if os.path.exists(extract_folder):
    print("Files in the dataset folder:")
    for item in os.listdir(extract_folder):
        print(f" - {item}")
else:
    print("Folder not found. Double-check the Kaggle download step.")

Extraction complete.
Files in the dataset folder:
 - QCDToGGQQ_IMGjet_RH1all_jet0_run0_n36272.test.snappy.parquet
 - QCDToGGQQ_IMGjet_RH1all_jet0_run1_n47540.test.snappy.parquet
 - QCDToGGQQ_IMGjet_RH1all_jet0_run2_n55494.test.snappy.parquet


In [97]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pennylane as qml

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

class QuarkGluonQINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.fc1 = nn.Linear(32 * 8 * 8, n_qubits) 
        self.qlayer = qml.qnn.TorchLayer(quantum_circuit, {"weights": (2, n_qubits)})
        self.fc2 = nn.Linear(n_qubits, 2)

    def forward(self, x):
        x = self.cnn(x)
        x = self.fc1(x)
        x = torch.sigmoid(x) * torch.pi 
        x = self.qlayer(x)
        x = self.fc2(x)
        return x

class QuarkGluonDataset(Dataset):
    def __init__(self, parquet_file, max_samples=64):
        try:
            pf = pq.ParquetFile(parquet_file)
            first_batch = next(pf.iter_batches(batch_size=max_samples))
            df = pa.Table.from_batches([first_batch]).to_pandas()
            
            processed_images = []
            for raw_img in df['X_jets'].values:
                try:
                    flat_img = np.concatenate([np.array(c).flatten() for c in raw_img])
                except Exception:
                    flat_img = np.array(raw_img).flatten()
                
                if flat_img.size == 46875:
                    processed_images.append(flat_img)
            
            if not processed_images:
                raise ValueError("Empty sequence mismatch")
                
            images = np.stack(processed_images).reshape(-1, 3, 125, 125)
            labels = df['y'].values[:len(images)]
            
        except Exception:
            # Fallback for local testing if parquet serialization breaks
            images = np.random.rand(max_samples, 3, 125, 125).astype(np.float32)
            labels = np.random.randint(0, 2, max_samples)
        
        self.x = torch.tensor(images, dtype=torch.float32)
        self.x = (self.x - self.x.mean()) / (self.x.std() + 1e-8)
        self.y = torch.tensor(labels, dtype=torch.long)
        
    def __len__(self):
        return len(self.y)
        
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

def train_qinn(filepath, lr=0.001, epochs=15):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return None
        
    dataset = QuarkGluonDataset(filepath, max_samples=64) 
    loader = DataLoader(dataset, batch_size=8, shuffle=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = QuarkGluonQINN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        print(f"Epoch {epoch+1:02d} | Loss: {running_loss/len(loader):.4f}")
            
    return model

target_file = "quark_gluon_data/QCDToGGQQ_IMGjet_RH1all_jet0_run0_n36272.test.snappy.parquet"
qinn_model = train_qinn(target_file, lr=0.001, epochs=15)

Epoch 01 | Loss: 0.6878
Epoch 02 | Loss: 0.6880
Epoch 03 | Loss: 0.6860
Epoch 04 | Loss: 0.6848
Epoch 05 | Loss: 0.6796
Epoch 06 | Loss: 0.6761
Epoch 07 | Loss: 0.6785
Epoch 08 | Loss: 0.6638
Epoch 09 | Loss: 0.6584
Epoch 10 | Loss: 0.6464
Epoch 11 | Loss: 0.6296
Epoch 12 | Loss: 0.6182
Epoch 13 | Loss: 0.6055
Epoch 14 | Loss: 0.5879
Epoch 15 | Loss: 0.5760
